# DiffusionGemma — NVIDIA Free API

Runs `google/diffusiongemma-26b-a4b-it` via [build.nvidia.com](https://build.nvidia.com) — no GPU required.

**Before starting:** make sure `.env` exists with `NVIDIA_API_KEY=nvapi-...`

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "google/diffusiongemma-26b-a4b-it"

client = OpenAI(base_url=BASE_URL, api_key=os.environ["NVIDIA_API_KEY"])
print("Client ready")

## Basic inference

In [ ]:
prompt = "Explain diffusion language models in 3 sentences."

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=256,
)
print(resp.choices[0].message.content)

## Streaming

In [ ]:
prompt = "Write a short poem about masked language modeling."

with client.chat.completions.stream(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=256,
) as stream:
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
print()

## Interactive playground

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_box = widgets.Textarea(
    value="Summarize the key differences between autoregressive and diffusion language models.",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="80px"),
)
max_tokens_slider = widgets.IntSlider(value=256, min=64, max=1024, step=64, description="Max tokens:")
temperature_slider = widgets.FloatSlider(value=0.7, min=0.0, max=1.5, step=0.05, description="Temperature:")
run_button = widgets.Button(description="Generate", button_style="primary")
output_area = widgets.Output()

def on_run(_):
    with output_area:
        clear_output()
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt_box.value}],
            max_tokens=max_tokens_slider.value,
            temperature=temperature_slider.value,
        )
        print(resp.choices[0].message.content)
        u = resp.usage
        print(f"\n[tokens: {u.prompt_tokens} in / {u.completion_tokens} out]")

run_button.on_click(on_run)
display(prompt_box, max_tokens_slider, temperature_slider, run_button, output_area)

## Batch comparison

In [ ]:
prompts = [
    "What is a diffusion model?",
    "How does masked diffusion differ from denoising diffusion?",
    "Give one advantage of diffusion LLMs over GPT-style models.",
]

for i, p in enumerate(prompts, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": p}],
        max_tokens=128,
    )
    print(f"--- [{i}] {p}")
    print(resp.choices[0].message.content)
    print()